# Helpdesk training — improved (henryk / U-ED-LSTM)

Sibling of `src/notebooks/training_variational_dropout/Helpdesk/full_enc_dec_lstm_gn.ipynb` that implements one of the three improvements from `improvements.md`:

- **§2 Training fix.** Two compounding changes:
  - **§2.a Linear teacher-forcing annealing.** Replace the stepwise decay in `Trainer.train_model` with linear annealing from `1.0` (epoch 1) to `0.3` (final epoch) via the `LinearTFTrainer` subclass below.
  - **§2.b Class-weighted activity CE.** Replace plain CE on the activity head with inverse-frequency weighted CE (computed from the augmented train set's suffix targets) via the `WeightedLoss` subclass below. The weighting only fires for the activity head; the resource head and the uncertainty MSEs are unchanged.

Loads the **augmented** pickles produced by `Helpdesk_full_loader_improved.ipynb` (which also implements §1 and gates §3 — Insert-ticket prepend augmentation and Variant-index removal). Per-epoch checkpoints overwrite the same `saving_path` (no numbered files).

## Imports

In [1]:
import importlib
import os
import sys
import pickle
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm

sys.path.insert(0, '../../../..')
sys.path.insert(0, '../../../../..')

## Data — load augmented train/val from `encoded_data/improved/`

In [2]:
PROJECT_ROOT = Path('../../../../..').resolve()

file_path_train = PROJECT_ROOT / 'encoded_data' / 'improved' / 'helpdesk_all_5_train.pkl'
file_path_val = PROJECT_ROOT / 'encoded_data' / 'improved' / 'helpdesk_all_5_val.pkl'

helpdesk_train_dataset = torch.load(str(file_path_train), weights_only=False)
helpdesk_val_dataset = torch.load(str(file_path_val), weights_only=False)
print('Train type:', type(helpdesk_train_dataset))
print('Val type:  ', type(helpdesk_val_dataset))

Train type: <class 'event_log_loader.new_event_log_loader.EventLogDataset'>
Val type:   <class 'event_log_loader.new_event_log_loader.EventLogDataset'>


In [3]:
helpdesk_all_categories = helpdesk_train_dataset.all_categories
helpdesk_all_categories_cat = helpdesk_all_categories[0]
helpdesk_all_categories_num = helpdesk_all_categories[1]

for i, cat in enumerate(helpdesk_all_categories_cat):
    print(f'cat[{i}] {cat[0]!r}: {cat[1]} classes')
for i, num in enumerate(helpdesk_all_categories_num):
    print(f'num[{i}] {num[0]!r}')

cat[0] 'Activity': 16 classes
cat[1] 'Resource': 24 classes
cat[2] 'Variant index': 175 classes
cat[3] 'seriousness': 3 classes
cat[4] 'customer': 361 classes
cat[5] 'product': 23 classes
cat[6] 'responsible_section': 9 classes
cat[7] 'seriousness_2': 6 classes
cat[8] 'service_level': 6 classes
cat[9] 'service_type': 6 classes
cat[10] 'support_section': 8 classes
cat[11] 'workgroup': 6 classes
num[0] 'case_elapsed_time'
num[1] 'event_elapsed_time'
num[2] 'day_in_week'
num[3] 'seconds_in_day'


## Encoder / decoder features

In [4]:
enc_feat_cat = [cat[0] for cat in helpdesk_all_categories_cat]
enc_feat_num = [num[0] for num in helpdesk_all_categories_num]
enc_feat = [enc_feat_cat, enc_feat_num]
print('Input features encoder:', enc_feat)

dec_feat_cat = ['Activity', 'Resource']
dec_feat_num = ['case_elapsed_time', 'event_elapsed_time']
dec_feat = [dec_feat_cat, dec_feat_num]
print('Features decoder:      ', dec_feat)

encoding_data = {
    'all_categories': helpdesk_all_categories,
    'encoding_features': enc_feat,
    'decoding_features': dec_feat,
}
with open('encoding_data.pkl', 'wb') as f:
    pickle.dump(encoding_data, f)

Input features encoder: [['Activity', 'Resource', 'Variant index', 'seriousness', 'customer', 'product', 'responsible_section', 'seriousness_2', 'service_level', 'service_type', 'support_section', 'workgroup'], ['case_elapsed_time', 'event_elapsed_time', 'day_in_week', 'seconds_in_day']]
Features decoder:       [['Activity', 'Resource'], ['case_elapsed_time', 'event_elapsed_time']]


## §2.b — compute class weights from the suffix targets in the train set

In [5]:
SUFFIX_SPLIT = 4
ACTIVITY_INDEX = 0
ACTIVITY_NUM_CLASSES = helpdesk_all_categories_cat[ACTIVITY_INDEX][1]
ACTIVITY_LABEL_DICT = helpdesk_all_categories_cat[ACTIVITY_INDEX][2]

counts = torch.zeros(ACTIVITY_NUM_CLASSES)
for cats, _, _ in DataLoader(helpdesk_train_dataset, batch_size=256, num_workers=0):
    suffix_targets = cats[ACTIVITY_INDEX][:, -SUFFIX_SPLIT:].flatten().long()
    counts += torch.bincount(suffix_targets, minlength=ACTIVITY_NUM_CLASSES).float()

raw_weights = 1.0 / (counts + 1.0)
raw_weights[0] = 0.0
n_nonzero = (raw_weights > 0).sum().clamp(min=1)
activity_class_weights = raw_weights * n_nonzero / raw_weights.sum().clamp(min=1e-12)

print('Per-class suffix-target frequency and inverse-frequency weight:')
idx_to_name = {v: k for k, v in ACTIVITY_LABEL_DICT.items()}
for idx in range(ACTIVITY_NUM_CLASSES):
    name = idx_to_name.get(idx, f'<idx {idx}>')
    print(f'  {idx:2d}  count={int(counts[idx]):6d}  weight={activity_class_weights[idx].item():.4f}  {name!r}')

Per-class suffix-target frequency and inverse-frequency weight:
   0  count=     0  weight=0.0000  '<idx 0>'
   1  count=  1947  weight=0.0038  'Assign seriousness'
   2  count= 10877  weight=0.0007  'Closed'
   3  count=   136  weight=0.0546  'Create SW anomaly'
   4  count=     4  weight=1.4972  'DUPLICATE'
   5  count= 53503  weight=0.0001  'EOS'
   6  count=     4  weight=1.4972  'INVALID'
   7  count=     0  weight=7.4861  'Insert ticket'
   8  count=     4  weight=1.4972  'RESOLVED'
   9  count=   215  weight=0.0347  'Require upgrade'
  10  count=    33  weight=0.2202  'Resolve SW anomaly'
  11  count=  9506  weight=0.0008  'Resolve ticket'
  12  count=     3  weight=1.8715  'Schedule intervention'
  13  count=  6353  weight=0.0012  'Take in charge ticket'
  14  count=     8  weight=0.8318  'VERIFIED'
  15  count=  2635  weight=0.0028  'Wait'


## Model

In [6]:
import model.dropout_uncertainty_enc_dec_LSTM.dropout_uncertainty_model
importlib.reload(model.dropout_uncertainty_enc_dec_LSTM.dropout_uncertainty_model)
from model.dropout_uncertainty_enc_dec_LSTM.dropout_uncertainty_model import DropoutUncertaintyEncoderDecoderLSTM

seq_len_pred = 4
hidden_size = 128
num_layers = 4
dropout = 0.0

with open('model_params_4layers.pkl', 'wb') as f:
    pickle.dump({
        'data_set_categories': helpdesk_all_categories,
        'enc_feat': enc_feat,
        'dec_feat': dec_feat,
        'seq_len_pred': seq_len_pred,
        'hidden_size': hidden_size,
        'num_layers': num_layers,
        'dropout': dropout,
    }, f)

model_obj = DropoutUncertaintyEncoderDecoderLSTM(
    data_set_categories=helpdesk_all_categories,
    enc_feat=enc_feat,
    dec_feat=dec_feat,
    seq_len_pred=seq_len_pred,
    hidden_size=hidden_size,
    num_layers=num_layers,
    dropout=dropout,
)

Data set categories:  ([('Activity', 16, {'Assign seriousness': 1, 'Closed': 2, 'Create SW anomaly': 3, 'DUPLICATE': 4, 'EOS': 5, 'INVALID': 6, 'Insert ticket': 7, 'RESOLVED': 8, 'Require upgrade': 9, 'Resolve SW anomaly': 10, 'Resolve ticket': 11, 'Schedule intervention': 12, 'Take in charge ticket': 13, 'VERIFIED': 14, 'Wait': 15}), ('Resource', 24, {'EOS': 1, 'Value 1': 2, 'Value 10': 3, 'Value 11': 4, 'Value 12': 5, 'Value 13': 6, 'Value 14': 7, 'Value 15': 8, 'Value 16': 9, 'Value 17': 10, 'Value 18': 11, 'Value 19': 12, 'Value 2': 13, 'Value 20': 14, 'Value 21': 15, 'Value 22': 16, 'Value 3': 17, 'Value 4': 18, 'Value 5': 19, 'Value 6': 20, 'Value 7': 21, 'Value 8': 22, 'Value 9': 23}), ('Variant index', 175, {'1.0': 1, '10.0': 2, '100.0': 3, '103.0': 4, '104.0': 5, '107.0': 6, '109.0': 7, '11.0': 8, '110.0': 9, '112.0': 10, '113.0': 11, '114.0': 12, '115.0': 13, '117.0': 14, '118.0': 15, '12.0': 16, '120.0': 17, '122.0': 18, '123.0': 19, '124.0': 20, '125.0': 21, '126.0': 22, '1

## §2.b — `WeightedLoss` subclass

In [7]:
import loss.losses
importlib.reload(loss.losses)
from loss.losses import Loss


class WeightedLoss(Loss):
    def __init__(self, weights_for_num_classes=None):
        super().__init__()
        self.weights_for_num_classes = weights_for_num_classes or {}

    def _weight_for(self, pred_logits):
        nc = pred_logits.shape[-1]
        w = self.weights_for_num_classes.get(nc)
        return w.to(pred_logits.device) if w is not None else None

    def standard_cross_entropy(self, pred_logits, targets):
        weight = self._weight_for(pred_logits)
        if weight is None:
            return super().standard_cross_entropy(pred_logits, targets)
        CEL = torch.nn.CrossEntropyLoss(reduction='none', weight=weight)
        pred_logits_p = pred_logits.permute(1, 2, 0)
        L = CEL(input=pred_logits_p, target=targets)
        return torch.mean(torch.mean(L, dim=1))

    def loss_attenuation_cross_entropy(self, pred_logits, pred_logvars, T, targets):
        weight = self._weight_for(pred_logits)
        if weight is None:
            return super().loss_attenuation_cross_entropy(pred_logits, pred_logvars, T, targets)
        CEL = torch.nn.CrossEntropyLoss(reduction='none', weight=weight)
        std = torch.sqrt(torch.exp(pred_logvars))
        L = 0
        for _ in range(T):
            noise = torch.randn_like(pred_logits)
            noisy = (pred_logits + std * noise).permute(1, 2, 0)
            L = L + CEL(input=noisy, target=targets)
        L = (1 / T) * L
        return torch.mean(torch.mean(L, dim=1))


loss_obj = WeightedLoss(weights_for_num_classes={ACTIVITY_NUM_CLASSES: activity_class_weights})
print(f'WeightedLoss configured for class-count {ACTIVITY_NUM_CLASSES}')

WeightedLoss configured for class-count 16


## §2.a — `LinearTFTrainer` subclass (per-epoch save overwrites the same file)

In [8]:
import trainer.trainer
importlib.reload(trainer.trainer)
from trainer.trainer import Trainer


class LinearTFTrainer(Trainer):
    def __init__(self, *args, tf_start=1.0, tf_end=0.3, **kwargs):
        super().__init__(*args, **kwargs)
        self.tf_start = float(tf_start)
        self.tf_end = float(tf_end)
        if self.epochs > 1:
            self.tf_schedule = [
                self.tf_start - (self.tf_start - self.tf_end) * e / (self.epochs - 1)
                for e in range(self.epochs)
            ]
        else:
            self.tf_schedule = [self.tf_end]
        print(f'Linear TF schedule: {self.tf_schedule[0]:.3f} (epoch 1) -> '
              f'{self.tf_schedule[-1]:.3f} (epoch {self.epochs})')

    def train_model(self):
        self.model.train()
        train_attenuated_losses, val_losses, val_attenuated_losses = [], [], []

        val_dataloader = DataLoader(
            dataset=self.data_val, batch_size=self.mini_batches,
            shuffle=self.shuffle, num_workers=4, pin_memory=True,
        )

        for epoch in tqdm(range(self.epochs)):
            train_dataloader = DataLoader(
                dataset=self.data_train, batch_size=self.mini_batches,
                shuffle=self.shuffle, num_workers=4, pin_memory=True,
            )

            epoch_cat_loss, epoch_num_loss = {}, {}
            epoch_loss = 0.0
            num_batches_per_epoch = 0.0

            self.teacher_forcing_ratio = self.tf_schedule[epoch]

            for i, train_data in enumerate(train_dataloader):
                cats, nums, _ = train_data
                prefixes_cat = [cat[:, :-self.suffix_data_split_value].to(self.device) for cat in cats]
                prefixes_num = [num[:, :-self.suffix_data_split_value].to(self.device) for num in nums]
                prefixes = [prefixes_cat, prefixes_num]
                suffixes_cat = [cat[:, -self.suffix_data_split_value:].to(self.device) for cat in cats]
                suffixes_num = [num[:, -self.suffix_data_split_value:].to(self.device) for num in nums]
                suffixes = [suffixes_cat, suffixes_num]

                if self.use_gradnorm:
                    all_losses_dict, loss_value = self.train_epoch_gradnorm(prefixes=prefixes, suffixes=suffixes)
                else:
                    all_losses_dict, loss_value = self.train_epoch(prefixes=prefixes, suffixes=suffixes)
                cat_losses_dict, num_losses_dict = all_losses_dict

                for fn, v in cat_losses_dict.items():
                    epoch_cat_loss[fn] = epoch_cat_loss.get(fn, 0.0) + v.item()
                for fn, v in num_losses_dict.items():
                    epoch_num_loss[fn] = epoch_num_loss.get(fn, 0.0) + v.item()
                epoch_loss += loss_value.item()
                num_batches_per_epoch += 1

            for fn in epoch_cat_loss:
                epoch_cat_loss[fn] /= num_batches_per_epoch
            for fn in epoch_num_loss:
                epoch_num_loss[fn] /= num_batches_per_epoch
            epoch_loss_train = epoch_loss / num_batches_per_epoch

            current_lr = self.scheduler.optimizer.param_groups[0]['lr']
            tqdm.write(
                f'Epoch [{epoch+1}/{self.epochs}], LR: {current_lr}, '
                f'TF: {self.teacher_forcing_ratio:.4f}'
            )
            tqdm.write(f'Training: Avg Attenuated Training Loss: {epoch_loss_train:.4f}')
            train_attenuated_losses.append(epoch_loss_train)

            (epoch_cat_loss_val_std, epoch_cat_loss_val_unc,
             epoch_num_loss_val_std, epoch_num_loss_val_unc,
             epoch_loss_val_std, epoch_loss_val_unc) = self.validation_epoch(val_dataloader=val_dataloader)

            tqdm.write(f'Validation: Avg Standard Validation Loss: {epoch_loss_val_std:.4f}')
            tqdm.write(f'Validation: Avg Attenuated Validation Loss: {epoch_loss_val_unc:.4f}')
            val_losses.append(epoch_loss_val_std)
            val_attenuated_losses.append(epoch_loss_val_unc)

            self.writer.add_scalars('Hyperparameter:', {
                'Learning Rate': current_lr,
                'Teacher Forcing Ratio': self.teacher_forcing_ratio,
            }, epoch + 1)
            self.writer.add_scalars('Total Losses', {
                'Training Total': epoch_loss_train,
                'Standard Validation Total': epoch_loss_val_std,
                'Uncertainty Validation Total': epoch_loss_val_unc,
            }, epoch + 1)
            for fn in epoch_cat_loss:
                self.writer.add_scalars('Categorical Feature Losses', {
                    f'Training {fn}': epoch_cat_loss[fn],
                    f'Standard Validation {fn}': epoch_cat_loss_val_std[fn],
                    f'Uncertainty Validation {fn}': epoch_cat_loss_val_unc[fn],
                }, epoch + 1)
            for fn in epoch_num_loss:
                self.writer.add_scalars('Numerical Feature Losses', {
                    f'Training {fn}': epoch_num_loss[fn],
                    f'Standard Validation {fn}': epoch_num_loss_val_std[fn],
                    f'Uncertainty Validation {fn}': epoch_num_loss_val_unc[fn],
                }, epoch + 1)
            if self.use_gradnorm:
                write_weights = self.gn_weights.data.cpu().numpy()
                feature_losses = list(epoch_cat_loss.keys()) + list(epoch_num_loss.keys())
                for i, fn in enumerate(feature_losses):
                    self.writer.add_scalars('Gradnorm values', {
                        f'Gradnorm Weight {fn}': write_weights[i],
                    }, epoch + 1)

            tqdm.write(f'Validation Loss for Scheduler: {epoch_loss_val_std:.4f}')
            self.scheduler.step(epoch_loss_val_std)

            # Overwrite the same file each epoch (no per-epoch numbered checkpoints)
            if self.save_model_n_th_epoch and (epoch + 1) % self.save_model_n_th_epoch == 0:
                tqdm.write(f'saving model to {self.saving_path}')
                self.model.save(self.saving_path)

        print('Training complete.')
        self.model.save(self.saving_path)
        tqdm.write(f'Model saved to path: {self.saving_path}')
        return train_attenuated_losses, val_losses, val_attenuated_losses

## Training configuration

In [9]:
writer = SummaryWriter(comment='Helpdesk_improved_henryk')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

learning_rate = 1e-4
optimizer = torch.optim.Adam(params=model_obj.parameters(), lr=learning_rate, weight_decay=0)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5, min_lr=1e-10)

num_epochs = 100
batch_size = 128
regularization_term = 1e-4
shuffle = True
teacher_forcing_ratio = 1.0

optimize_values = {
    'regularization_term': regularization_term,
    'optimizer': optimizer,
    'scheduler': scheduler,
    'epochs': num_epochs,
    'mini_batches': batch_size,
    'shuffle': shuffle,
    'teacher_forcing_ratio': teacher_forcing_ratio,
}

suffix_data_split_value = SUFFIX_SPLIT

use_gradnorm = True
gn_alpha = 1.5
gn_learning_rate = 1e-4
number_tasks = len(dec_feat[0]) + len(dec_feat[1])
gradNorm = {
    'use_gradnorm': use_gradnorm,
    'number_tasks': number_tasks,
    'gn_alpha': gn_alpha,
    'gn_learning_rate': gn_learning_rate,
}

saving_path = 'Helpdesk_full_grad_norm_improved_henryk.pkl'

trainer_obj = LinearTFTrainer(
    device=device,
    model=model_obj,
    data_train=helpdesk_train_dataset,
    data_val=helpdesk_val_dataset,
    loss_obj=loss_obj,
    log_normal_loss_num_feature=[],
    optimize_values=optimize_values,
    suffix_data_split_value=suffix_data_split_value,
    writer=writer,
    gradnorm_values=gradNorm,
    save_model_n_th_epoch=1,
    saving_path=saving_path,
    tf_start=1.0,
    tf_end=0.3,
)

Device: cpu
Device:  cpu
Model:  DropoutUncertaintyEncoderDecoderLSTM(
  (embeddings_enc): ModuleList(
    (0): Embedding(16, 8)
    (1): Embedding(24, 9)
    (2): Embedding(175, 29)
    (3): Embedding(3, 3)
    (4): Embedding(361, 43)
    (5): Embedding(23, 9)
    (6): Embedding(9, 5)
    (7-9): 3 x Embedding(6, 4)
    (10): Embedding(8, 5)
    (11): Embedding(6, 4)
  )
  (encoder): DropoutUncertaintyLSTMEncoder(
    (embeddings): ModuleList(
      (0): Embedding(16, 8)
      (1): Embedding(24, 9)
      (2): Embedding(175, 29)
      (3): Embedding(3, 3)
      (4): Embedding(361, 43)
      (5): Embedding(23, 9)
      (6): Embedding(9, 5)
      (7-9): 3 x Embedding(6, 4)
      (10): Embedding(8, 5)
      (11): Embedding(6, 4)
    )
    (first_layer): DropoutUncertaintyLSTMCell(
      (Wi): Linear(in_features=131, out_features=128, bias=True)
      (Ui): Linear(in_features=128, out_features=128, bias=True)
      (Wf): Linear(in_features=131, out_features=128, bias=True)
      (Uf): Linea

## Train

In [ ]:
train_attenuated_losses, val_losses, val_attenuated_losses = trainer_obj.train_model()

  0%|          | 0/100 [00:00<?, ?it/s]/Users/philippeichhorn/.local/share/virtualenvs/XAI-Probabilistic_Suffix_Prediction_U-ED-L-p9C7deWr/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
  0%|          | 0/100 [00:54<?, ?it/s]

Epoch [1/100], LR: 0.0001, TF: 1.0000
Training: Avg Attenuated Training Loss: 3.1060


  1%|          | 1/100 [01:10<1:56:19, 70.50s/it]

Validation: Avg Standard Validation Loss: 3.2263
Validation: Avg Attenuated Validation Loss: 1.5639
Validation Loss for Scheduler: 3.2263
saving model to Helpdesk_full_grad_norm_improved_henryk.pkl


  1%|          | 1/100 [02:19<1:56:19, 70.50s/it]

...

Epoch [11/100], LR: 0.0001, TF: 0.9293
Training: Avg Attenuated Training Loss: -0.6886


 11%|█         | 11/100 [12:25<1:27:03, 58.69s/it]

Validation: Avg Standard Validation Loss: 3.2785
Validation: Avg Attenuated Validation Loss: 140.1284
Validation Loss for Scheduler: 3.2785
saving model to Helpdesk_full_grad_norm_improved_henryk.pkl


 11%|█         | 11/100 [13:11<1:27:03, 58.69s/it]

Epoch [12/100], LR: 0.0001, TF: 0.9222
Training: Avg Attenuated Training Loss: -1.5444


## Loss curves

In [ ]:
import matplotlib.pyplot as plt

epochs_axis = range(1, num_epochs + 1)
plt.figure()
plt.plot(epochs_axis, train_attenuated_losses, label='Training Attenuated Loss')
plt.plot(epochs_axis, val_losses, label='Validation Standard Loss')
plt.plot(epochs_axis, val_attenuated_losses, label='Validation Attenuated Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Helpdesk improved (henryk) — training/validation loss')
plt.legend()
plt.show()